In [5]:
#Extensions - potential

#ADF Test for mean reversion - for stocks and indices that trend up, use differences to model the residual
#For Big markets that do mean reversion etc
# 1. sell above MA, buy below MA
# 2. Sell if Day/Week is +ve, Buy if -ve
# 3. Combine both

#High movements - Black Swan moves
# Recent surprise, quick jumps - a glut or a squeeze which may see retracements
# Above, shorter term analysis eg Open/close gap analyses esp with inefficiencies in new datasets....
# Open and Close seem to be nearer, High/Low are wilder. Anything there?

#Arbitrage: Cross-mkts
#Analysis on High/Low differences etc between cross-markets

#New tickers esp small tickers that move wildly
#Adding new tickers at the bottom and others? Like BTC, JPYx etc

In [6]:
# !pip install hurst
# !pip install arch 
# !pip install lightgbm 
# !pip install scikit-learn 
# !pip install pmdarima
# !pip install shap
# !pip install numpy pandas

In [7]:
# Import the libraries
import pyautogui
import time
import pandas as pd
import numpy as np
import plotly.graph_objs as go
import os
import itertools

from datetime import date

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)  # or None for no limit

In [8]:
#Name the Forex and Metals tickers
Curr = [
    #Currencies
    "USDCAD", "USDCHF", "USDCNH", "USDCZK", "USDHUF", "USDJPY", "USDMXN", "USDNOK", "USDPLN", "USDSEK",
    "USDSGD", "USDTHB", "USDTRY", "USDZAR", "USDIDR", "USDINR", 
    'USDX.a','EURX',
    "EURUSD", "GBPUSD","NZDUSD"
    
    #Metals
    ,"XAGUSD.a","XAUUSD.a", "XPTUSD.a",
    
    #ETFs
    'AUS200.a','US30.a','US500.a','UK100.a','NAS100.a','EUSTX50.a','SPA35.a','JPN225.a', 'GER40.a','HK50.a',
    'NETH25.a','CN50.a','SCI25.a','SWI20.a','FRA40.a', #'NOR25.a'
    
#     # US Shares tickers
    "AMD.US-24", "BABA.US-24", "GOOG.US-24", "AMZN.US-24", "AAPL.US-24", "BAC.US-24", "CAT.US-24",
#     "CVX.US-24", "C.US-24", "XOM.US-24", "F.US-24", "GM.US-24", "HPQ.US-24", "IBM.US-24", "INTC.US-24",
    "JPM.US-24", "JNJ.US-24", "MCD.US-24", "META.US-24", "MSFT.US-24", "NKE.US-24", "NVDA.US-24",
    "NFLX.US-24", "ORCL.US-24", "PFE.US-24", "PG.US-24", "SLB.US-24", "SNAP.US-24", "TSLA.US-24", "WMT.US-24",
#     "BA.US-24", "KO.US-24", "DIS.US-24", "UNH.US-24", "VZ.US-24", "RTX.US-24", "V.US-24"
    
    #Commodities and Crypto
    # Swap positive for short
    "Wheat",
#     "Cocoa.a", "Cotton.a", "Sugar.a", "Corn.a",
    #Swap negative for short
    "BTCUSD", "SpotBrent"
#     "Coffee.a", "Soybeans.a", "LDSugar.a", "Cattle.a"
    
    # AU Shares tickers - vet these before
#     "A2M.AU"
#     "AGL.AU", "AIA.AU", "AIZ.AU", "ALD.AU", "ALQ.AU", "ALX.AU", "AMC.AU"

]

# Curr = ['USDCHF','USDCNH','USDSGD','GBPUSD','US500.a','UK100.a','GER40.a','CN50.a']

In [9]:
#To run the modules created
import sys
sys.path.append("..")                 # if needed to find project_pkg
import project_pkg.dld as dld_module  # import module object
from importlib import reload
reload(dld_module)

# call function/class inside the module
dld_module.dld(Curr)
# Curr2 = ['USDJPY', 'EURUSD','US30.a']
# dld_module.dld(Curr2)

import project_pkg.utils as utils

In [10]:
# Analysis on the datasets
base_path = r"C:\Users\nitis\Documents\Forex\Data\New data"
res1 = {} #Define the dict

from project_pkg.utils import (
    numpy_slope, Run, Filter, df_div, determine_direction_and_momentum, analyse,
    get_hurst, get_MFDFA, calculate_hurst, rolling_hurst
)

#Loop through all currencies, len(daily)
for name in Curr:
    path = fr"{base_path}\{name}.csv"
    df = pd.read_csv(path, sep='\t', engine='python')
    res1[name] = df

In [11]:
#Creating new dfs for sans-USD
#Gold and Silver
#One-way symbols:
#Others and crosses by inverting. Easy
symbols = ["XAGUSD.a","XAUUSD.a", 
          "EURUSD", "GBPUSD", "AUDUSD", "NZDUSD",
          "USDCAD", "USDCHF", "USDJPY", "USDMXN", "USDSGD"]
res2 = {} #Define the dict
res2 = df_div(symbols)
# print(res2)
# print(res2.keys())
# print(res2['EURCHF'])

In [12]:
import pandas as pd
import numpy as np
import project_pkg.filters_2 as filt

# Dictionary to hold the final cockpit rows for reporting
dashboard_records = []

fin_res = res1 | res2
for name, df in fin_res.items():
    try:
        # 1. Run the master matrix calculator on the symbol's dataframe
        processed_df = filt.build_confluence_matrix(df)
        
        # 2. Extract the latest available row (today's live state)
        # Drop rows where MAs are still warming up
        valid_rows = processed_df.dropna()
        
        if valid_rows.empty:
            print(f"⚠️ [Skipping {name}]: Insufficient historical data to warm up MAs.")
            continue
            
        latest_live_row = valid_rows.iloc[-1]
        
        # 3. Pull the live count values directly from the processed state
        COUNT_LONG  = latest_live_row["COUNT_LONG"]
        COUNT_SHORT = latest_live_row["COUNT_SHORT"]
        COUNT_HOLD  = latest_live_row["COUNT_HOLD"]
        
        # Logic Routing Engine
        if COUNT_LONG >= 12 and COUNT_SHORT <=3:
            action = "Long"
        elif COUNT_SHORT >= 12 and COUNT_LONG <=3:
            action = "Short"
        else:
            action = "No"  # Safe default fallback for exact statistical ties
            
        # 4. Pack the structural data row into our master database
        dashboard_records.append({
            "Symbol": name,
            "Spot Price": latest_live_row["<CLOSE>"],
            "Count Long": COUNT_LONG,
            "Count Short": COUNT_SHORT,
            "Count Hold": COUNT_HOLD,
            "Engine Action": action,
            "As Of Date": pd.to_datetime(latest_live_row["<DATE>"]).strftime("%Y-%m-%d")
        })
        
        # print(f"✅ Processed {name.ljust(8)} | Date: {pd.to_datetime(latest_live_row['<DATE>']).strftime('%Y-%m-%d')} | Action: {action}")
        
    except Exception as e:
        print(f"❌ [Error processing {name}]: {str(e)}")

# Final results cockpit compilation
if dashboard_records:
    master_dashboard_df = pd.DataFrame(dashboard_records)
    print(f"\n🚀 Execution Complete: Successfully matrix-scored {len(master_dashboard_df)} symbols.")
else:
    print("\n❌ Execution Complete: No symbols were successfully matrix-scored.")

⚠️ [Skipping USDIDR]: Insufficient historical data to warm up MAs.
⚠️ [Skipping USDINR]: Insufficient historical data to warm up MAs.

🚀 Execution Complete: Successfully matrix-scored 62 symbols.


In [13]:
#Option to analyse manually
import pandas as pd
# path = fr"C:\Users\nitis\Desktop\List_06 Jun.txt"
# master_dashboard_df2 = pd.read_fwf(path) #For txt file load

from datetime import date

master_dashboard_df2 = master_dashboard_df[master_dashboard_df['Engine Action'] != 'No']

# Get today's date
today = date.today()

# Format the date as 'DD Mon' (e.g., '02 Nov')
date_string = today.strftime('%d %b')

filename = f"List_{date_string}.txt"

# Create the text output string
text_output = master_dashboard_df.to_string(index=False, header=True)

# 2. DEFINE THE FULL PATH
import os
base_path = r'C:\Users\nitis\Desktop'
base_path2 = r'C:\Users\nitis\Desktop\Fin stuff\Week output'

full_path = os.path.join(base_path, filename)
full_path2 = os.path.join(base_path2, filename)

# Write the string directly to the file
with open(full_path, 'w', encoding='utf-8') as f:
    f.write(text_output)

# Write the string directly to the file
with open(full_path2, 'w', encoding='utf-8') as f:
    f.write(text_output)

In [14]:
%load_ext autoreload
%autoreload 2

# To run the modules created
from project_pkg.model import (
    calculate_technical_indicators,
    run_walk_forward_validation_tuned, 
    predict_next_day_scenarios
)

import pandas as pd
import warnings

# IMPORT VALUEWARNING SO PYTHON RECOGNIZES IT
from statsmodels.tools.sm_exceptions import ValueWarning

# 1. Force Python to silence the specific statsmodels base warnings
warnings.filterwarnings("ignore", category=ValueWarning, module="statsmodels")
warnings.filterwarnings("ignore", category=FutureWarning, module="statsmodels")

# Quick catch-all backup if the module path names differ slightly in your environment:
warnings.filterwarnings("ignore", message=".*No supported index is available.*")

# 1. Create a manual filtered DataFrame for your specific tech basket
# manual_tickers = [
#     "GOOG.US-24", "AMZN.US-24", "AAPL.US-24", "AMD.US-24", "NVDA.US-24", 
#     "MSFT.US-24", "TSLA.US-24", "META.US-24", "ORCL.US-24", "NAS100.a"
#     'SpotBrent','Wheat','XPTUSD.a'
#     'XAGUSD.a','NAS100.a','XAUUSD.a'
# ]
# master_dashboard_df2 = pd.DataFrame(manual_tickers, columns=['Symbol'])

# 2. Loop through the unique currencies in the filtered DataFrame
for currency in master_dashboard_df2['Symbol']:
    # Ensure we handle any potential missing/NaN values safely
    if pd.isna(currency):
        continue
        
    # Construct the file path using an f-string
    file_path = fr"C:\Users\nitis\Documents\Forex\Data\New data\{currency}.csv"
    
    print(f"\n==================================================")
    print(f"PROCESSING ASSET PIPELINE FOR: {currency}")
    print(f"==================================================")
    
    # Read the CSV file
    df_mod = pd.read_csv(file_path, sep='\t', engine='python')

    # --- RUN THE TUNED REGRESSION BACKTEST ---
    # train_window=300 aligns with your production engine historical training lookback
    (
        historical_mae, 
        historical_sharpe, 
        historical_sortino, 
        historical_max_dd, 
        bench_sharpe,
        ranked_features
    ) = run_walk_forward_validation_tuned(
        df=df_mod, 
        train_window=300,
        retrain_interval=100
    )

    print("\n" + "="*50)
    print("           TUNED HISTORICAL BACKTEST RESULTS      ")
    print("==================================================")
    print(f"Regression Mean Absolute Error (MAE): {historical_mae:.5f}")
    print("==================================================")
    print(f"Strategy Sharpe Ratio: {historical_sharpe:.2f}  (Benchmark B&H: {bench_sharpe:.2f})") 
    print("==================================================")
    print(f"Strategy Sortino Ratio: {historical_sortino:.2f}") 
    print("==================================================")
    print(f"Max Peak-to-Trough Drawdown:   {historical_max_dd:.2%}") 
    print("==================================================")
    print("\n           COMPLETE FEATURE IMPORTANCE RANKINGS     ")
    print("==================================================")
    for rank, (feature, importance) in enumerate(ranked_features, 1):
         print(f"Rank {rank:02d}: {feature:<22} -> Relative Impact: {importance:.2f}%")
    print("==================================================")

    # --- GENERATE INTRADAY DEVELOPMENT SCENARIOS FOR TOMORROW ---
    # This runs the 6-scenario decision rubric and outputs your interactive trade board
    predict_next_day_scenarios(df_mod, train_window=300)


PROCESSING ASSET PIPELINE FOR: USDJPY

           TUNED HISTORICAL BACKTEST RESULTS      
Regression Mean Absolute Error (MAE): 0.00446
Strategy Sharpe Ratio: 0.77  (Benchmark B&H: 0.60)
Strategy Sortino Ratio: 1.05
Max Peak-to-Trough Drawdown:   -2.14%

           COMPLETE FEATURE IMPORTANCE RANKINGS     
Rank 01: price_zscore_50        -> Relative Impact: 13.40%
Rank 02: volatility_5           -> Relative Impact: 11.72%
Rank 03: vol_acceleration       -> Relative Impact: 11.30%
Rank 04: dist_MA_5              -> Relative Impact: 9.82%
Rank 05: RSI                    -> Relative Impact: 7.30%
Rank 06: Stochastic_14          -> Relative Impact: 6.98%
Rank 07: ROC_10                 -> Relative Impact: 6.51%
Rank 08: dist_MA_20             -> Relative Impact: 6.44%
Rank 09: NTR_5                  -> Relative Impact: 6.34%
Rank 10: volume_ratio           -> Relative Impact: 5.85%
Rank 11: CO_spread              -> Relative Impact: 4.95%
Rank 12: returns                -> Relative Impact


             DAILY MODEL REGIME RISK & DURABILITY MATRIX             
 REGIME ENVIRONMENT               │ STRAT SHARPE │ SORTINO │ MAX DD  │ BENCH SHARPE
──────────────────────────────────┼──────────────┼─────────┼─────────┼─────────────
 Raw Historical Baseline Data     │     2.61     │   6.54  │  -2.0% │     3.46
 Flavor 1: Liquidity Black Hole   │     0.58     │   0.53  │ -11.3% │     --
 Flavor 2: Structural Bear Market │     2.18     │   3.06  │  -8.0% │     --
 Flavor 3: High-Volatility Grind  │     3.82     │   6.15  │  -4.1% │     --
 Flavor 4: The Gold Rush (Bubble) │     1.56     │   2.04  │  -7.2% │     --


PROCESSING ASSET PIPELINE FOR: US30.a

           TUNED HISTORICAL BACKTEST RESULTS      
Regression Mean Absolute Error (MAE): 0.00612
Strategy Sharpe Ratio: -0.06  (Benchmark B&H: 1.03)
Strategy Sortino Ratio: -0.07
Max Peak-to-Trough Drawdown:   -5.30%

           COMPLETE FEATURE IMPORTANCE RANKINGS     
Rank 01: vol_acceleration       -> Relative Impact: 14.86%
Ran

 [METRIC VALUES]  Trend Z: +1.93 (Med: +1.39) | Mom Dist: +0.0253 (Med: +0.0188)
                  Vol Spike: -12.1% (Med: -1.8%) | ATR Ratio: 0.85 (Med: 0.98)
----------------------------------------------------------------------------------------------------------------------------------
 [STATE: MATURE INSTITUTIONAL GRIND]
   -> Profile : Elevated trend and momentum, but trading volume and true range are drying up.
   -> Bias    : Reduce global position sizes. This is a low-liquidity grinding environment prone to false breakouts.
----------------------------------------------------------------------------------------------------------------------------------

Calculating historical macro-durability metrics for engine audit...

             DAILY MODEL REGIME RISK & DURABILITY MATRIX             
 REGIME ENVIRONMENT               │ STRAT SHARPE │ SORTINO │ MAX DD  │ BENCH SHARPE
──────────────────────────────────┼──────────────┼─────────┼─────────┼─────────────
 Raw Historical Baseli


             DAILY MODEL REGIME RISK & DURABILITY MATRIX             
 REGIME ENVIRONMENT               │ STRAT SHARPE │ SORTINO │ MAX DD  │ BENCH SHARPE
──────────────────────────────────┼──────────────┼─────────┼─────────┼─────────────
 Raw Historical Baseline Data     │     0.86     │   1.06  │  -3.9% │     1.27
 Flavor 1: Liquidity Black Hole   │     0.58     │   0.68  │  -7.2% │     --
 Flavor 2: Structural Bear Market │     0.68     │   0.75  │  -9.3% │     --
 Flavor 3: High-Volatility Grind  │     3.65     │   6.01  │  -4.7% │     --
 Flavor 4: The Gold Rush (Bubble) │    -0.39     │  -0.37  │ -18.3% │     --


PROCESSING ASSET PIPELINE FOR: SPA35.a

           TUNED HISTORICAL BACKTEST RESULTS      
Regression Mean Absolute Error (MAE): 0.00727
Strategy Sharpe Ratio: -0.04  (Benchmark B&H: 1.56)
Strategy Sortino Ratio: -0.05
Max Peak-to-Trough Drawdown:   -3.92%

           COMPLETE FEATURE IMPORTANCE RANKINGS     
Rank 01: volatility_5           -> Relative Impact: 10.34%
Ra

 6. High-Vol Indecision Cross     │ 1106.35000 │ +0.4399%    │ BUY LONG  +0.36x   │ 1120.75900   │ 1096.74400

                                          DYNAMIC 4-METRIC DATA-DRIVEN REGIME COCKPIT
----------------------------------------------------------------------------------------------------------------------------------
 [METRIC VALUES]  Trend Z: +2.45 (Med: +1.03) | Mom Dist: +0.0429 (Med: +0.0085)
                  Vol Spike: +2.1% (Med: -0.6%) | ATR Ratio: 0.65 (Med: 0.99)
----------------------------------------------------------------------------------------------------------------------------------
 [STATE: LIQUIDITY DISTRIBUTION / INSIDER SELLING]
   -> Profile : Price looks technically strong or flat, volume is spiking heavily, but actual price volatility is being compressed.
   -> Bias    : Major players are absorbing buy orders without letting price move up. Exercise extreme caution on long trends.
------------------------------------------------------------------------


           TUNED HISTORICAL BACKTEST RESULTS      
Regression Mean Absolute Error (MAE): 0.02527
Strategy Sharpe Ratio: 0.23  (Benchmark B&H: 1.01)
Strategy Sortino Ratio: 0.30
Max Peak-to-Trough Drawdown:   -11.53%

           COMPLETE FEATURE IMPORTANCE RANKINGS     
Rank 01: ROC_10                 -> Relative Impact: 12.90%
Rank 02: volatility_5           -> Relative Impact: 11.00%
Rank 03: volume_ratio           -> Relative Impact: 9.50%
Rank 04: dist_MA_5              -> Relative Impact: 8.87%
Rank 05: NTR_5                  -> Relative Impact: 7.32%
Rank 06: dist_MA_20             -> Relative Impact: 6.89%
Rank 07: volume_spread_ratio    -> Relative Impact: 6.76%
Rank 08: vol_acceleration       -> Relative Impact: 6.14%
Rank 09: RSI                    -> Relative Impact: 6.13%
Rank 10: price_zscore_50        -> Relative Impact: 5.84%
Rank 11: returns                -> Relative Impact: 5.31%
Rank 12: Stochastic_14          -> Relative Impact: 4.73%
Rank 13: HL_spread             

 5. Compressed Neutral Chop       │ 314.92000  │ +0.2001%    │ BUY LONG  +0.09x   │ 325.62700    │ 307.78200
 6. High-Vol Indecision Cross     │ 314.92000  │ -0.2858%    │ SELL SHORT -0.14x  │ 304.21300    │ 322.05800

                                          DYNAMIC 4-METRIC DATA-DRIVEN REGIME COCKPIT
----------------------------------------------------------------------------------------------------------------------------------
 [METRIC VALUES]  Trend Z: +2.34 (Med: +0.91) | Mom Dist: +0.0667 (Med: +0.0142)
                  Vol Spike: +4.9% (Med: +0.2%) | ATR Ratio: 0.88 (Med: 0.97)
----------------------------------------------------------------------------------------------------------------------------------
 [STATE: LIQUIDITY DISTRIBUTION / INSIDER SELLING]
   -> Profile : Price looks technically strong or flat, volume is spiking heavily, but actual price volatility is being compressed.
   -> Bias    : Major players are absorbing buy orders without letting price move up. Exerc


           TUNED HISTORICAL BACKTEST RESULTS      
Regression Mean Absolute Error (MAE): 0.01958
Strategy Sharpe Ratio: -0.38  (Benchmark B&H: 0.27)
Strategy Sortino Ratio: -0.46
Max Peak-to-Trough Drawdown:   -17.09%

           COMPLETE FEATURE IMPORTANCE RANKINGS     
Rank 01: volume_ratio           -> Relative Impact: 10.22%
Rank 02: vol_acceleration       -> Relative Impact: 9.48%
Rank 03: dist_MA_5              -> Relative Impact: 7.69%
Rank 04: CO_spread              -> Relative Impact: 7.54%
Rank 05: dist_MA_20             -> Relative Impact: 7.54%
Rank 06: HL_spread              -> Relative Impact: 7.52%
Rank 07: RSI                    -> Relative Impact: 7.42%
Rank 08: NTR_5                  -> Relative Impact: 7.39%
Rank 09: volume_spread_ratio    -> Relative Impact: 7.36%
Rank 10: returns                -> Relative Impact: 5.98%
Rank 11: price_zscore_50        -> Relative Impact: 5.88%
Rank 12: volatility_5           -> Relative Impact: 5.49%
Rank 13: Stochastic_14        

In [17]:
import pandas as pd 
import project_pkg.trad_econ as te

# https://tradingeconomics.com/matrix

# Define the exact layout schema matching your visible screen columns
COLUMN_NAMES = [
    "Country",
    "GDP",
    "GDP_Growth",
    "Interest_Rate",
    "Inflation_Rate",
    "Unemployment_rate",
    "Gov_Budget",
    "Gov_Debt_to_GDP",
    "Current_Account_to_GDP",
    "Population"  
]

try:
    # 1. Read directly from your computer's clipboard
    df_raw = pd.read_clipboard(header=None, names=COLUMN_NAMES)
    df_raw["Country"] = df_raw["Country"].str.strip()
    df_matrix = df_raw.set_index("Country")
    
    # 2. Force data to numeric so blank fields or strings don't crash the ranks
    for col in df_matrix.columns:
        df_matrix[col] = pd.to_numeric(df_matrix[col], errors='coerce')
    
    # 3. Append the Final Score as the LAST column on the far right
    df_matrix["Final_Macro_Score"] = te.calculate_macro_scores(df_matrix)
    
    # 4. Sort the entire DataFrame by that new column
    df_sorted = df_matrix.sort_values(by="Final_Macro_Score", ascending=False)
    
    print(f"[+] Successfully processed {len(df_sorted)} countries.")
    display(df_sorted)

except Exception as e:
    print(f"[!] Execution Failed. Did you remember to copy the table rows first?")
    print(f"Details: {e}")

[+] Successfully processed 16 countries.


,GDP,GDP_Growth,Interest_Rate,Inflation_Rate,Unemployment_rate,Gov_Budget,Gov_Debt_to_GDP,Current_Account_to_GDP,Population,Final_Macro_Score
Country,,,,,,,,,,
South Korea,1872,1.8,2.50,3.20,2.80,-3.9,49.00,6.60,51.68,0.69
China,19498,1.3,3.00,1.00,5.10,-6.5,99.20,3.70,1405.00,0.63
Russia,2561,NaN,14.25,6.00,2.10,-2.6,18.30,2.00,146.00,0.59
Japan,4435,0.5,1.00,1.50,2.50,-2.3,249.00,4.70,123.00,0.57
India,3956,1.9,5.25,3.93,5.50,-4.4,81.92,-0.60,1421.00,0.57
Germany,5051,0.3,NaN,2.30,6.30,-2.7,63.50,4.50,83.50,0.55
United Kingdom,4003,0.6,3.75,2.80,4.90,-4.3,94.30,-2.40,69.49,0.54
Australia,1799,0.3,4.35,4.00,4.40,-1.6,18.80,-3.70,27.70,0.52
Mexico,1833,-0.6,6.50,3.37,2.80,-3.9,45.40,-0.40,132.00,0.51


In [16]:
# # Other shares, softs, hards, etc

# # AU Shares tickers
# au_share_tickers = [
#     "A2M.AU", "ABC.AU", "AGL.AU", "AIA.AU", "AIZ.AU", "ALD.AU", "ALQ.AU", "ALU.AU", "ALX.AU", "AMC.AU",
#     "AMP.AU", "ANN.AU", "ANZ.AU", "APA.AU", "APE.AU", "APX.AU", "ARG.AU", "ARB.AU", "ASB.AU", "ASX.AU",
#     "AST.AU", "AUB.AU", "AWB.AU", "AWL.AU", "AWN.AU", "AWC.AU", "BEN.AU", "BGA.AU", "BHP.AU", "BIN.AU",
#     "BKL.AU", "BKW.AU", "BLD.AU", "BOQ.AU", "BPT.AU", "BRG.AU", "BSL.AU", "BVS.AU", "BWP.AU", "BXB.AU",
#     "CAR.AU", "CBA.AU", "CCP.AU", "CCL.AU", "CGC.AU", "CGF.AU", "CHC.AU", "CIP.AU", "CIM.AU", "CIA.AU",
#     "CLW.AU", "CMW.AU", "CNU.AU", "COL.AU", "COE.AU", "COH.AU", "CPU.AU", "CQR.AU", "CRN.AU", "CSL.AU",
#     "CSR.AU", "CTD.AU", "CUV.AU", "CWN.AU", "CWY.AU", "DHG.AU", "DMP.AU", "DOW.AU", "DDR.AU", "DEG.AU",
#     "DRR.AU", "DXS.AU", "EBO.AU", "EHE.AU", "EHL.AU", "ELD.AU", "EHE.AU", "EML.AU", "EVT.AU", "EVN.AU",
#     "FBU.AU", "FLT.AU", "FMG.AU", "FPH.AU", "GEM.AU", "GNC.AU", "GMG.AU", "GOR.AU", "GOZ.AU", "GPT.AU",
#     "GQG.AU", "GUD.AU", "GWA.AU", "HDN.AU", "HLS.AU", "HMC.AU", "HUB.AU", "HVN.AU", "IAG.AU", "IFT.AU",
#     "IEL.AU", "IFM.AU", "IFL.AU", "ILU.AU", "INA.AU", "INC.AU", "IRE.AU", "IPL.AU", "IPH.AU", "IVC.AU",
#     "JBH.AU", "JHX.AU", "JHG.AU", "JIN.AU", "KGN.AU", "LNK.AU", "LIS.AU", "LLC.AU", "LOV.AU", "LTR.AU",
#     "LYC.AU", "MFG.AU", "MIN.AU", "MGR.AU", "MMS.AU", "MPL.AU", "MP1.AU", "MQG.AU", "MTS.AU", "MYX.AU",
#     "NAN.AU", "NAB.AU", "NEC.AU", "NEU.AU", "NHC.AU", "NHF.AU", "NIC.AU", "NEM.AU", "NCM.AU", "NEA.AU",
#     "NUF.AU", "NSR.AU", "NWH.AU", "NXT.AU", "NWL.AU", "ORA.AU", "ORG.AU", "ORI.AU", "OML.AU", "ORE.AU",
#     "OZL.AU", "PDL.AU", "PDN.AU", "PER.AU", "PLS.AU", "PME.AU", "PMV.AU", "PNI.AU", "PNV.AU", "PPK.AU",
#     "PPT.AU", "PRN.AU", "PRU.AU", "PTM.AU", "PXA.AU", "QAN.AU", "QBE.AU", "QUB.AU", "RBL.AU", "REA.AU",
#     "REH.AU", "REG.AU", "RHC.AU", "RIO.AU", "RMD.AU", "RRL.AU", "RSG.AU", "RWC.AU", "S32.AU", "SAR.AU",
#     "SBM.AU", "SCA.AU", "SCG.AU", "SCP.AU", "SDG.AU", "SDF.AU", "SEK.AU", "SFR.AU", "SGM.AU", "SHL.AU",
#     "SIQ.AU", "SKC.AU", "SKI.AU", "SLC.AU", "SLR.AU", "SML.AU", "SMR.AU", "SOL.AU", "SPK.AU", "SSG.AU",
#     "STO.AU", "SUL.AU", "SUN.AU", "SVW.AU", "SWM.AU", "SYD.AU", "SXL.AU", "TAH.AU", "TCL.AU", "TGR.AU",
#     "TLS.AU", "TLX.AU", "TNE.AU", "TPG.AU", "TWE.AU", "URW.AU", "VCX.AU", "VEA.AU", "VOC.AU", "VNT.AU",
#     "VUK.AU", "WBC.AU", "WEB.AU", "WES.AU", "WMC.AU", "WOR.AU", "WOW.AU", "WPL.AU", "WPR.AU", "WSA.AU",
#     "WTC.AU", "XRO.AU", "YAL.AU", "Z1P.AU"
# ]

# # Combine all tickers and sort
# all_tickers = sorted(au_share_tickers + us_share_tickers + forex_metals_tickers)

# # Create DataFrame and export to CSV
# df = pd.DataFrame(all_tickers, columns